# NB29 · Performance & Memory

`advanced/` · notebook 29 of 32 · 26 questions + a mini-project

Fast code starts with a measurement, not a hunch. This notebook gives you
the tools to measure (`timeit`, `perf_counter`, `cProfile`, `tracemalloc`)
and the knowledge to act on what you find: which containers are cheap for
which jobs, why `"".join` beats `+=`, how generators and `__slots__` save
memory, and how Python decides when an object can be freed (reference
counting, the cycle collector `gc`, and `weakref`).

**About timing numbers.** Timings change every run and on every machine,
so no question here prints one. Instead you'll *count* things that don't
change: comparisons, function calls, bytes allocated. That's also a
senior habit. A count of operations is the honest version of Big-O
(NB19), and it never lies because the laptop was busy.

Everything runs in CPython, the standard Python. Other implementations
(PyPy, the free-threaded build from NB27) manage memory differently, so
reference-counting details can differ there.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB19: Big-O
- [ ] NB28: testing

## What you'll practise

- Measuring with timeit, perf_counter and cProfile
- Costs of built-in operations; choosing the right container
- join vs += for strings; generators to save memory
- __slots__, caching, local variable lookups
- References, reference counting, gc, cycles, weakref
- sys.getsizeof and its limits; array vs list for numbers

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB29-Q01 · L1 · Measuring · Write

Use `timeit.timeit` to time the statement `"sum(range(100))"` run
`1000` times. Don't print the time (it changes every run). Print the
**type name** of the result and whether it is greater than zero.

```text
float True
```

**Starting code (already in the cell below):**

```python
import timeit
```

<details><summary><b>Hint</b></summary>

`timeit.timeit(statement_text, number=...)` returns the total seconds as a number. `type(x).__name__` gives a type's name as text.

</details>

<details><summary><b>Expected output</b></summary>

```text
float True
```

</details>

<details><summary><b>Solution</b></summary>

```python
seconds = timeit.timeit("sum(range(100))", number=1000)
print(type(seconds).__name__, seconds > 0)
```

`timeit.timeit` runs the statement `number` times and returns the **total** seconds as a `float`. To get the time per run, divide by `number`. It uses the most precise clock available and switches off the garbage collector while timing, so the result is steadier than timing by hand.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
best = min(timeit.repeat("sum(range(100))", number=1000, repeat=5))
print(type(best).__name__, best > 0)
```

Seniors use `timeit.repeat` and take the **minimum**. Slower runs are caused by other programs stealing the CPU, never by your code getting faster, so the fastest run is the best estimate of the true cost.

</details>

In [ ]:
import timeit

# your code here


### NB29-Q02 · L1 · References · Predict

Two names, one list. `sys.getrefcount` reports how many references
point at an object. Predict the output.

**What does this print?**

```python
import sys

basket = ["eggs"]
before = sys.getrefcount(basket)
backup = basket
backup.append("milk")
print(sys.getrefcount(basket) - before)
print(basket, backup is basket)
```

<details><summary><b>Hint</b></summary>

`backup = basket` copies nothing. It adds one more name for the *same* list.

</details>

<details><summary><b>Expected output</b></summary>

```text
1
['eggs', 'milk'] True
```

</details>

<details><summary><b>Solution</b></summary>

Assignment never copies. `backup = basket` makes a second reference to the same list, so the reference count goes up by `1`, and appending through either name changes the one list both names share. CPython frees an object the moment its reference count drops to zero. That is *reference counting*, Python's main memory manager.

</details>

<details><summary><b>Senior dev solution</b></summary>

The absolute number from `getrefcount` is noisy (the call itself adds a temporary reference, and the interpreter holds some of its own), so seniors only ever look at the **difference** between two readings.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB29-Q03 · L1 · sys.getsizeof · Write

Print two booleans on one line: is `small` smaller in memory than
`big`, and is the tuple `point_t` smaller than the list `point_l`
holding the same values? Use `sys.getsizeof`.

```text
True True
```

**Starting code (already in the cell below):**

```python
import sys

small = [1, 2, 3]
big = list(range(1000))
point_t = (3, 4)
point_l = [3, 4]
```

<details><summary><b>Hint</b></summary>

`sys.getsizeof(obj)` returns the object's size in bytes. Compare two sizes with `<`.

</details>

<details><summary><b>Expected output</b></summary>

```text
True True
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(sys.getsizeof(small) < sys.getsizeof(big),
      sys.getsizeof(point_t) < sys.getsizeof(point_l))
```

A list stores one pointer per item, so 1000 items need far more room than 3. A tuple is smaller than a list with the same items: it can never grow, so it needs no spare capacity and no separate storage block.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
list_bigger = sys.getsizeof(small) < sys.getsizeof(big)
tuple_leaner = sys.getsizeof(point_t) < sys.getsizeof(point_l)
print(list_bigger, tuple_leaner)
```

Named results document what each comparison proves. Seniors also reach for tuples for fixed records (coordinates, RGB colours) partly for this saving, but mostly because immutability prevents bugs.

</details>

In [ ]:
import sys

small = [1, 2, 3]
big = list(range(1000))
point_t = (3, 4)
point_l = [3, 4]

# your code here


---

## L2 · Easy

### NB29-Q04 · L2 · Measuring · Write

Time how long `sum(range(1_000_000))` takes using `time.perf_counter()`.
Print the sum, then whether the elapsed time is greater than zero (never
the time itself).

```text
499999500000
True
```

**Starting code (already in the cell below):**

```python
import time
```

<details><summary><b>Hint</b></summary>

Read the clock before and after the work. The difference is the elapsed time in seconds.

</details>

<details><summary><b>Expected output</b></summary>

```text
499999500000
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
start = time.perf_counter()
total = sum(range(1_000_000))
elapsed = time.perf_counter() - start
print(total)
print(elapsed > 0)
```

`perf_counter()` is a high-resolution clock meant for measuring durations. Its value on its own means nothing; only the difference between two readings does. Use it when you want to time one real run of something, such as a whole report or a request.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
start = time.perf_counter()
total = sum(range(1_000_000))
elapsed_s = time.perf_counter() - start
print(total)
print(elapsed_s > 0)
```

Never use `time.time()` for durations: it follows the wall clock, which can jump when the system syncs its time. `perf_counter` only moves forward. Putting the unit in the name (`elapsed_s`) stops seconds and milliseconds getting mixed up.

</details>

In [ ]:
import time

# your code here


### NB29-Q05 · L2 · sys.getsizeof · Predict

One short string, one huge string. Predict the output.

**What does this print?**

```python
import sys

short = ["hi"]
huge = ["hi" * 10_000]
print(sys.getsizeof(short) == sys.getsizeof(huge))
print(sys.getsizeof("hi") < sys.getsizeof("hi" * 10_000))
```

<details><summary><b>Hint</b></summary>

A list holds *references* to its items, not the items themselves. What does `getsizeof` measure: the list, or everything it points to?

</details>

<details><summary><b>Expected output</b></summary>

```text
True
True
```

</details>

<details><summary><b>Solution</b></summary>

`sys.getsizeof` is **shallow**: it measures the list object (its header and one pointer per item) and ignores what the pointers lead to. Both lists hold one pointer, so they are the same size. The strings themselves differ hugely, which the second line shows.

</details>

<details><summary><b>Senior dev solution</b></summary>

To estimate the real footprint of nested data, seniors either walk the structure adding sizes (watching for shared objects) or, better, measure actual allocations with `tracemalloc` (Q18).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB29-Q06 · L2 · join vs += · Write

Turn the sensor `readings` into one comma-separated line using
`str.join`, then print it.

```text
21.5,22.0,22.4,21.9,23.1
```

**Starting code (already in the cell below):**

```python
readings = [21.5, 22.0, 22.4, 21.9, 23.1]
```

<details><summary><b>Hint</b></summary>

`join` only accepts strings, so each reading has to become text first.

</details>

<details><summary><b>Expected output</b></summary>

```text
21.5,22.0,22.4,21.9,23.1
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(",".join(str(r) for r in readings))
```

`",".join(...)` glues strings together with a comma between each pair. It works out the final length once and copies each piece once, so its cost grows linearly. The generator expression converts each float to text as `join` asks for it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print(",".join(map(str, readings)))
```

`map(str, readings)` is the idiomatic spelling when you apply an existing function unchanged. For a real CSV file, seniors use the `csv` module (NB13), which also handles commas *inside* values.

</details>

In [ ]:
readings = [21.5, 22.0, 22.4, 21.9, 23.1]

# your code here


### NB29-Q07 · L2 · weakref · Error

A weak reference to a plain list. Which error does this raise?

**Which error does this raise?**

```python
import weakref

cart = [3, 1, 2]
watch = weakref.ref(cart)
print(watch())
```

<details><summary><b>Hint</b></summary>

Not every type can be weakly referenced. Lists, dicts, ints and strings are kept as small as possible.

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: built-in `list`, `dict`, `int`, `str` and `tuple` objects don't carry the hidden slot a weak reference needs, to keep them small. Instances of your own classes do have it, and so do functions and sets.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import weakref

class Cart(list):
    pass

cart = Cart([3, 1, 2])
watch = weakref.ref(cart)
print(watch())
```

A one-line subclass of `list` (or `dict`) *can* be weakly referenced, because instances of Python classes get the weakref slot. That's the standard workaround.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB29-Q08 · L2 · Choosing containers · Write

Security gives you a **list** of banned user ids. Convert it to a set
**once**, then print `ok` or `banned` for each visitor, one per line.

```text
u104 ok
u777 banned
u215 ok
u902 banned
```

**Starting code (already in the cell below):**

```python
banned_list = ["u902", "u331", "u777", "u480"]
visitors = ["u104", "u777", "u215", "u902"]
```

<details><summary><b>Hint</b></summary>

`set(banned_list)` before the loop. Then `in` on the set inside the loop.

</details>

<details><summary><b>Expected output</b></summary>

```text
u104 ok
u777 banned
u215 ok
u902 banned
```

</details>

<details><summary><b>Solution</b></summary>

```python
banned = set(banned_list)
for user in visitors:
    if user in banned:
        print(user, "banned")
    else:
        print(user, "ok")
```

`in` on a list checks items one by one: O(n). `in` on a set jumps straight to the right slot using the item's hash: O(1) on average. Building the set costs one pass, so do it once, outside the loop. Building it inside the loop would cost a full pass per visitor.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
BANNED = frozenset(banned_list)
for user in visitors:
    print(user, "banned" if user in BANNED else "ok")
```

A `frozenset` signals "this lookup table never changes" and can't be edited by accident. The rule of thumb: a list for order, a set for membership, a dict for lookup by key.

</details>

In [ ]:
banned_list = ["u902", "u331", "u777", "u480"]
visitors = ["u104", "u777", "u215", "u902"]

# your code here


---

## L3 · Intermediate

### NB29-Q09 · L3 · Costs of operations · Write

Prove that `in` on a set beats a list **as n grows**, without timing
anything: count the `==` comparisons each one makes.

`Sku` counts every `==` call in `comparisons`. For each `n` in
`(100, 1_000, 10_000)`, build a list and a set of `Sku(0)` to
`Sku(n - 1)`, search each for a **new** `Sku(n - 1)` (the worst case
for the list), and print the two counts. Finally print whether the
list needed more than 100 times as many comparisons at `n = 10_000`.

```text
100 list: 100 set: 1
1000 list: 1000 set: 1
10000 list: 10000 set: 1
list needs 100x more: True
```

**Starting code (already in the cell below):**

```python
comparisons = 0

class Sku:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        global comparisons
        comparisons += 1
        return self.code == other.code

    def __hash__(self):
        return hash(self.code)
```

<details><summary><b>Hint</b></summary>

Reset `comparisons = 0` before each search and read it straight after. Search with a *new* object, not one from the list, so the comparison really happens.

</details>

<details><summary><b>Expected output</b></summary>

```text
100 list: 100 set: 1
1000 list: 1000 set: 1
10000 list: 10000 set: 1
list needs 100x more: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
for n in (100, 1_000, 10_000):
    items = [Sku(code) for code in range(n)]
    item_set = set(items)
    target = Sku(n - 1)
    comparisons = 0
    found = target in items
    list_count = comparisons
    comparisons = 0
    found = target in item_set
    set_count = comparisons
    print(n, "list:", list_count, "set:", set_count)
print("list needs 100x more:", list_count > 100 * set_count)
```

The list walks every item until it finds a match, so the count grows with `n`: O(n). The set hashes the target, jumps to one slot, and only calls `==` on an item whose hash matches: one comparison whatever `n` is, O(1). Counting operations gives the same answer on every machine, every run. A timing would not.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def count_eq(container, target):
    global comparisons
    comparisons = 0
    target in container
    return comparisons

for n in (100, 1_000, 10_000):
    items = [Sku(code) for code in range(n)]
    list_count = count_eq(items, Sku(n - 1))
    set_count = count_eq(set(items), Sku(n - 1))
    print(n, "list:", list_count, "set:", set_count)
print("list needs 100x more:", list_count > 100 * set_count)
```

A helper removes the copy-pasted reset-search-read steps. The idea behind it is the important part: instrument the operation you care about and count it. Counting is how you test performance claims in a unit test (NB28) without flaky timings.

</details>

In [ ]:
comparisons = 0

class Sku:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        global comparisons
        comparisons += 1
        return self.code == other.code

    def __hash__(self):
        return hash(self.code)

# your code here


### NB29-Q10 · L3 · join vs += · Refactor

This builds a stock report by adding to a string in a loop. Rewrite it
to collect the lines and join them once.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
stock = {"apples": 12, "pears": 0, "plums": 7}
report = ""
for name in stock:
    report += name + ": " + str(stock[name]) + "\n"
print(report, end="")
```

<details><summary><b>Hint</b></summary>

Build a list of line strings, then `"\n".join(lines)`. Watch the final newline.

</details>

<details><summary><b>Expected output</b></summary>

```text
apples: 12
pears: 0
plums: 7
```

</details>

<details><summary><b>Solution</b></summary>

```python
stock = {"apples": 12, "pears": 0, "plums": 7}
lines = []
for name, qty in stock.items():
    lines.append(f"{name}: {qty}")
print("\n".join(lines))
```

Strings are immutable, so every `+=` builds a brand-new string and copies everything so far into it. Over many lines that is O(n²) copying. Collecting pieces in a list and joining once copies each character once: O(n). CPython sometimes optimises `+=` in place, but only in specific situations, so you can't rely on it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
stock = {"apples": 12, "pears": 0, "plums": 7}
print("\n".join(f"{name}: {qty}" for name, qty in stock.items()))
```

One expression, no mutable accumulator, and the shape of each line is visible in the f-string. When the output goes to a file, seniors skip the big string entirely and `write` lines as they go.

</details>

In [ ]:
stock = {"apples": 12, "pears": 0, "plums": 7}
report = ""
for name in stock:
    report += name + ": " + str(stock[name]) + "\n"
print(report, end="")


### NB29-Q11 · L3 · cProfile · Write

Profile one call of `build_report(stock)` with `cProfile` and print how
many times `format_row` was called, using the numbers `pstats` collects.

```text
format_row 4
```

**Starting code (already in the cell below):**

```python
import cProfile
import pstats

def format_row(name, qty):
    return f"{name:<8}{qty:>4}"

def build_report(stock):
    lines = []
    for name, qty in stock.items():
        lines.append(format_row(name, qty))
    return "\n".join(lines)

stock = {"apples": 12, "pears": 0, "plums": 7, "kiwis": 30}
```

<details><summary><b>Hint</b></summary>

`profiler = cProfile.Profile()`, then `enable()`, run the code, `disable()`. `pstats.Stats(profiler).stats` is a dict keyed by `(filename, line, function_name)`; each value starts with the call counts.

</details>

<details><summary><b>Expected output</b></summary>

```text
format_row 4
```

</details>

<details><summary><b>Solution</b></summary>

```python
profiler = cProfile.Profile()
profiler.enable()
report = build_report(stock)
profiler.disable()

stats = pstats.Stats(profiler)
for (filename, line, name), numbers in stats.stats.items():
    if name == "format_row":
        print(name, numbers[1])
```

`cProfile` records every function call while it is enabled: how many times each function ran and how long it took. Each value in `stats.stats` is `(primitive_calls, total_calls, own_time, cumulative_time, callers)`. Call counts are exact and repeatable; the times are not, which is why we print only the count.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with cProfile.Profile() as profiler:
    report = build_report(stock)

profile = pstats.Stats(profiler).get_stats_profile()
print("format_row", profile.func_profiles["format_row"].ncalls)
```

`Profile` is a context manager (NB24), so `with` guarantees profiling stops even if the code raises. `get_stats_profile()` gives named fields instead of positional tuples. Day to day, seniors run `python -m cProfile -s cumulative script.py` and read the top rows.

</details>

In [ ]:
import cProfile
import pstats

def format_row(name, qty):
    return f"{name:<8}{qty:>4}"

def build_report(stock):
    lines = []
    for name, qty in stock.items():
        lines.append(format_row(name, qty))
    return "\n".join(lines)

stock = {"apples": 12, "pears": 0, "plums": 7, "kiwis": 30}

# your code here


### NB29-Q12 · L3 · __slots__ · Error

`Reading` uses `__slots__` to save memory. Which error does this
raise?

**Which error does this raise?**

```python
class Reading:
    __slots__ = ("sensor", "value")

    def __init__(self, sensor, value):
        self.sensor = sensor
        self.value = value

r = Reading("t1", 21.5)
r.unit = "C"
print(r.unit)
```

<details><summary><b>Hint</b></summary>

With `__slots__`, an instance has room for exactly the listed attributes and nothing else.

</details>

<details><summary><b>Expected output</b></summary>

```text
AttributeError
```

</details>

<details><summary><b>Solution</b></summary>

`AttributeError`: `__slots__` replaces the per-instance `__dict__` with fixed storage for the named attributes only, so there is nowhere to put `unit`. That is the price of the memory saving: no adding attributes on the fly.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Reading:
    __slots__ = ("sensor", "value", "unit")

    def __init__(self, sensor, value, unit="C"):
        self.sensor = sensor
        self.value = value
        self.unit = unit

r = Reading("t1", 21.5)
print(r.unit)
```

Seniors list every attribute up front (with defaults set in `__init__`). In practice they often write `@dataclass(slots=True)` (NB21) and let the slots be generated from the fields.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB29-Q13 · L3 · gc and cycles · Predict

Two objects that point at each other, with the cycle collector
switched off for a moment. Predict the output.

**What does this print?**

```python
import gc
import weakref

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None

gc.disable()
a = Node("a")
b = Node("b")
a.partner = b
b.partner = a
watch = weakref.ref(a)

del a, b
print(watch() is None)
gc.collect()
print(watch() is None)
gc.enable()
```

<details><summary><b>Hint</b></summary>

After `del`, each object is still referenced by the *other* one. Does its reference count ever reach zero?

</details>

<details><summary><b>Expected output</b></summary>

```text
False
True
```

</details>

<details><summary><b>Solution</b></summary>

`False` then `True`. After `del a, b`, no name points at either node, but each still holds a reference to the other, so neither count reaches zero and reference counting can't free them. That is a *reference cycle*. The cycle collector (`gc`) exists to find groups of objects that only reference each other; `gc.collect()` runs it now, and the weak reference goes dead.

</details>

<details><summary><b>Senior dev solution</b></summary>

Normally `gc` runs automatically, so cycles are freed *eventually*, not immediately. When you need prompt cleanup (big buffers, open files), seniors break the cycle with a weak reference (Q21) or free resources explicitly with `with` (NB24) instead of relying on the collector.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB29-Q14 · L3 · Caching · Write

`shipping_cost` is slow (pretend it calls a remote service). Cache it
with `functools.lru_cache(maxsize=32)`, look up every zone in
`orders`, then print the total and the cache statistics.

```text
total: 61.5
CacheInfo(hits=3, misses=3, maxsize=32, currsize=3)
```

**Starting code (already in the cell below):**

```python
import functools

RATES = {"EU": 7.5, "US": 12.0, "ASIA": 15.0}

def shipping_cost(zone):
    return RATES[zone]

orders = ["EU", "US", "EU", "ASIA", "US", "EU"]
```

<details><summary><b>Hint</b></summary>

`functools.lru_cache(maxsize=32)(shipping_cost)` wraps an existing function. The wrapper has a `cache_info()` method.

</details>

<details><summary><b>Expected output</b></summary>

```text
total: 61.5
CacheInfo(hits=3, misses=3, maxsize=32, currsize=3)
```

</details>

<details><summary><b>Solution</b></summary>

```python
shipping_cost = functools.lru_cache(maxsize=32)(shipping_cost)
total = 0
for zone in orders:
    total += shipping_cost(zone)
print("total:", total)
print(shipping_cost.cache_info())
```

The first call for each zone is a *miss*: the real function runs and the result is stored. Every repeat is a *hit*, answered from the cache without running the function. Six lookups, three distinct zones: three misses, three hits. `cache_info()` is how you check a cache is earning its memory.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
@functools.lru_cache(maxsize=32)
def cached_cost(zone):
    return RATES[zone]

total = sum(cached_cost(zone) for zone in orders)
print("total:", total)
print(cached_cost.cache_info())
```

Seniors apply the cache as a decorator where the function is defined, and give `maxsize` a real limit so a long-running program can't grow the cache forever. A cache only suits *pure* functions: same arguments, same answer, no side effects.

</details>

In [ ]:
import functools

RATES = {"EU": 7.5, "US": 12.0, "ASIA": 15.0}

def shipping_cost(zone):
    return RATES[zone]

orders = ["EU", "US", "EU", "ASIA", "US", "EU"]

# your code here


### NB29-Q15 · L3 · Caching · Fix

This memoised Fibonacci should make only a few dozen calls, but it
prints a huge call count. Fix it so the memo is actually shared.

```text
6765 39
```

**Buggy code (copied into the cell below):**

```python
calls = 0

def fib(n, memo=None):
    global calls
    calls += 1
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n < 2:
        result = n
    else:
        result = fib(n - 1) + fib(n - 2)
    memo[n] = result
    return result

print(fib(20), calls)
```

<details><summary><b>Hint</b></summary>

Each recursive call receives `memo=None`. Where does it get its memo from?

</details>

<details><summary><b>Expected output</b></summary>

```text
6765 39
```

</details>

<details><summary><b>Solution</b></summary>

```python
calls = 0

def fib(n, memo=None):
    global calls
    calls += 1
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n < 2:
        result = n
    else:
        result = fib(n - 1, memo) + fib(n - 2, memo)
    memo[n] = result
    return result

print(fib(20), calls)
```

The recursive calls didn't pass `memo`, so each one made a fresh empty dict and the cache never helped: 21,891 calls, exponential growth. Passing the same dict down means each `n` is computed once and every repeat is a dictionary hit, so the count drops to 39. A cache only helps if every caller sees the same one.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import functools

@functools.cache
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

result = fib(20)
info = fib.cache_info()
print(result, info.hits + info.misses)
```

`@functools.cache` (NB11) does the bookkeeping for you and can't be forgotten on one branch, and `cache_info()` replaces the hand-made counter: 21 misses (real computations) plus 18 hits gives the same 39 calls. Measure what matters: 21 real computations instead of 21,891.

</details>

In [ ]:
calls = 0

def fib(n, memo=None):
    global calls
    calls += 1
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n < 2:
        result = n
    else:
        result = fib(n - 1) + fib(n - 2)
    memo[n] = result
    return result

print(fib(20), calls)


### NB29-Q16 · L3 · Local lookups · Write

Inside a loop, every global name is looked up in the module's
dictionary on each pass, while a local name is a direct slot access.
Use `dis.get_instructions` to print the sorted global names each
function loads (instructions whose `opname` is `"LOAD_GLOBAL"`; the
name is in `argval`), then check both give the same answer.

```text
slow reads globals: ['math', 'round']
fast reads globals: ['round']
same result: True
```

**Starting code (already in the cell below):**

```python
import dis
import math

def total_length_slow(points):
    total = 0.0
    for x, y in points:
        total += math.sqrt(x * x + y * y)
    return round(total, 2)

def total_length_fast(points, sqrt=math.sqrt):
    total = 0.0
    for x, y in points:
        total += sqrt(x * x + y * y)
    return round(total, 2)

points = [(3, 4), (6, 8), (1, 1)]
```

<details><summary><b>Hint</b></summary>

Loop over `dis.get_instructions(func)`, keep `instruction.argval` when `instruction.opname == "LOAD_GLOBAL"`, and `sorted()` the set you collect.

</details>

<details><summary><b>Expected output</b></summary>

```text
slow reads globals: ['math', 'round']
fast reads globals: ['round']
same result: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
slow_names = set()
for instruction in dis.get_instructions(total_length_slow):
    if instruction.opname == "LOAD_GLOBAL":
        slow_names.add(instruction.argval)
fast_names = set()
for instruction in dis.get_instructions(total_length_fast):
    if instruction.opname == "LOAD_GLOBAL":
        fast_names.add(instruction.argval)
print("slow reads globals:", sorted(slow_names))
print("fast reads globals:", sorted(fast_names))
print("same result:", total_length_slow(points) == total_length_fast(points))
```

`dis` shows the bytecode Python runs. The slow version loads the global `math` and then looks up its `.sqrt` attribute on **every** pass of the loop. The fast version binds `math.sqrt` once, as a default argument, so inside the loop `sqrt` is a local: the cheapest lookup there is. `round` is called once, outside the loop, so it doesn't matter.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def global_names(func):
    return sorted({i.argval for i in dis.get_instructions(func) if i.opname == "LOAD_GLOBAL"})

print("slow reads globals:", global_names(total_length_slow))
print("fast reads globals:", global_names(total_length_fast))
print("same result:", total_length_slow(points) == total_length_fast(points))
```

A set comprehension and a helper remove the duplication. The bigger senior point: this trick saves tens of nanoseconds per pass. Use it in a hot loop that profiling has pointed at, never as a habit; the default argument also leaks an odd parameter into the function's signature.

</details>

In [ ]:
import dis
import math

def total_length_slow(points):
    total = 0.0
    for x, y in points:
        total += math.sqrt(x * x + y * y)
    return round(total, 2)

def total_length_fast(points, sqrt=math.sqrt):
    total = 0.0
    for x, y in points:
        total += sqrt(x * x + y * y)
    return round(total, 2)

points = [(3, 4), (6, 8), (1, 1)]

# your code here


---

## L4 · Somewhat Difficult

### NB29-Q17 · L4 · Choosing containers · Refactor

This duplicate finder compares every pair of orders: O(n²). Rewrite it
as a single O(n) pass, keeping the output (duplicates in the order
they're first found) exactly the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
orders = ["A12", "B07", "A12", "C33", "B07", "D01", "A12"]
duplicates = []
for i in range(len(orders)):
    for j in range(i + 1, len(orders)):
        if orders[i] == orders[j] and orders[i] not in duplicates:
            duplicates.append(orders[i])
print(duplicates)
```

<details><summary><b>Hint</b></summary>

Remember what you've seen in a set. The second time you see a code, it's a duplicate. A second set stops you reporting the same code twice.

</details>

<details><summary><b>Expected output</b></summary>

```text
['A12', 'B07']
```

</details>

<details><summary><b>Solution</b></summary>

```python
orders = ["A12", "B07", "A12", "C33", "B07", "D01", "A12"]
seen = set()
reported = set()
duplicates = []
for code in orders:
    if code in seen and code not in reported:
        duplicates.append(code)
        reported.add(code)
    seen.add(code)
print(duplicates)
```

The nested loops compare about n²/2 pairs (21 for 7 orders, about 50 million for 10,000), and `not in duplicates` scans a list on top. The rewrite looks at each order once and only does O(1) set checks, so 10,000 orders cost about 10,000 steps. Same answer: a code becomes a duplicate the second time it appears, which matches the order the old version found them in.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import Counter

orders = ["A12", "B07", "A12", "C33", "B07", "D01", "A12"]
counts = Counter(orders)
print([code for code, n in counts.items() if n > 1])
```

`Counter` (NB15) counts in one O(n) pass and remembers first-seen order, so its keys come out A12, B07, C33, D01 and the filter keeps the repeated ones. The bigger win is choosing the right algorithm; no micro-optimisation of the nested loops could match it.

</details>

In [ ]:
orders = ["A12", "B07", "A12", "C33", "B07", "D01", "A12"]
duplicates = []
for i in range(len(orders)):
    for j in range(i + 1, len(orders)):
        if orders[i] == orders[j] and orders[i] not in duplicates:
            duplicates.append(orders[i])
print(duplicates)


### NB29-Q18 · L4 · Generators and memory · Write

Use `tracemalloc` to measure the **peak** memory of summing 100,000
squares two ways: from a list comprehension, and from a generator
expression. Print both sums, then whether the list's peak is more than
100 times the generator's (never the byte counts themselves).

```text
333328333350000 333328333350000
list peak > 100x generator peak: True
```

**Starting code (already in the cell below):**

```python
import tracemalloc
```

<details><summary><b>Hint</b></summary>

`tracemalloc.start()`, do the work, `tracemalloc.get_traced_memory()` returns `(current, peak)`, then `tracemalloc.stop()`. Measure each version separately.

</details>

<details><summary><b>Expected output</b></summary>

```text
333328333350000 333328333350000
list peak > 100x generator peak: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
tracemalloc.start()
list_total = sum([x * x for x in range(100_000)])
list_peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

tracemalloc.start()
gen_total = sum(x * x for x in range(100_000))
gen_peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(list_total, gen_total)
print("list peak > 100x generator peak:", list_peak > 100 * gen_peak)
```

The list comprehension builds all 100,000 squares before `sum` sees the first one, so memory peaks at the whole list plus every int in it (megabytes). The generator hands `sum` one square at a time and each is freed straight away, so the peak stays at a few hundred bytes. `tracemalloc` counts bytes actually allocated, so unlike timing it gives the same answer every run.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def peak_bytes(func):
    tracemalloc.start()
    try:
        result = func()
        return result, tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()

list_total, list_peak = peak_bytes(lambda: sum([x * x for x in range(100_000)]))
gen_total, gen_peak = peak_bytes(lambda: sum(x * x for x in range(100_000)))
print(list_total, gen_total)
print("list peak > 100x generator peak:", list_peak > 100 * gen_peak)
```

A helper with `try`/`finally` guarantees tracing stops even if the measured code fails (tracing slows everything down). Comparing peaks with a wide margin, rather than exact bytes, keeps the check stable across Python versions.

</details>

In [ ]:
import tracemalloc

# your code here


### NB29-Q19 · L4 · Measuring · Predict

How many times does `timeit` run the setup and the statement? Predict
the output.

**What does this print?**

```python
import timeit

log = {"setup": 0, "stmt": 0}

def prepare():
    log["setup"] += 1

def work():
    log["stmt"] += 1

timeit.timeit("work()", setup="prepare()", globals=globals(), number=5)
print(log)
timeit.repeat("work()", setup="prepare()", globals=globals(), number=5, repeat=3)
print(log)
```

<details><summary><b>Hint</b></summary>

`setup` runs once per timing round, not once per statement. `repeat` does several whole rounds.

</details>

<details><summary><b>Expected output</b></summary>

```text
{'setup': 1, 'stmt': 5}
{'setup': 4, 'stmt': 20}
```

</details>

<details><summary><b>Solution</b></summary>

One `timeit` round runs `setup` once, then the statement `number` times: `{'setup': 1, 'stmt': 5}`. `repeat=3` does three full rounds, adding 3 setups and 15 statements: `{'setup': 4, 'stmt': 20}`. Only the statement is timed, so put preparation (building test data) in `setup` and keep it out of the measurement.

</details>

<details><summary><b>Senior dev solution</b></summary>

A classic lying benchmark builds its input inside the timed statement and ends up measuring list construction instead of the operation. Seniors move every piece of preparation into `setup`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB29-Q20 · L4 · Benchmarks that lie · Predict

Someone "benchmarked" a generator and declared it instant. Count how
many squares were actually computed. Predict the output.

**What does this print?**

```python
import timeit

computed = 0

def square(x):
    global computed
    computed += 1
    return x * x

timeit.timeit("(square(x) for x in range(1000))", globals=globals(), number=100)
print(computed)
timeit.timeit("sum(square(x) for x in range(1000))", globals=globals(), number=100)
print(computed)
```

<details><summary><b>Hint</b></summary>

Creating a generator expression doesn't run its body. Something has to consume it.

</details>

<details><summary><b>Expected output</b></summary>

```text
0
100000
```

</details>

<details><summary><b>Solution</b></summary>

`0`, then `100000`. The first benchmark only *creates* 100 generator objects and never asks them for a value, so `square` never runs; it times the setup cost of a generator, not the work. Wrapping it in `sum(...)` consumes it: 1000 squares x 100 runs.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is the commonest lying micro-benchmark. Before trusting a timing, seniors check the benchmark does the real work (count it, as here, or compare the result) and measure the whole job, not a fragment.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB29-Q21 · L4 · weakref · Fix

Each folder keeps a list of its children, and each child points back
at its parent. With `gc` switched off (as it can be in a busy
program between collections), deleting the tree should free it
straight away, but it stays alive. Fix it with a weak reference for
the parent link, so the output becomes:

```text
parent: root
still alive: False
```

**Buggy code (copied into the cell below):**

```python
import gc
import weakref

class Folder:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = parent
        self.children = []
        if parent is not None:
            parent.children.append(self)

gc.disable()
root = Folder("root")
docs = Folder("docs", root)
print("parent:", docs.parent.name)
watch = weakref.ref(docs)
del root, docs
print("still alive:", watch() is not None)
gc.enable()
```

<details><summary><b>Hint</b></summary>

Store `weakref.ref(parent)` instead of `parent`. A weak reference is called to get the object back: `docs.parent()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
parent: root
still alive: False
```

</details>

<details><summary><b>Solution</b></summary>

```python
import gc
import weakref

class Folder:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = weakref.ref(parent) if parent is not None else None
        self.children = []
        if parent is not None:
            parent.children.append(self)

gc.disable()
root = Folder("root")
docs = Folder("docs", root)
print("parent:", docs.parent().name)
watch = weakref.ref(docs)
del root, docs
print("still alive:", watch() is not None)
gc.enable()
```

Parent to child and child to parent made a reference cycle, so after `del` neither count reached zero and only the cycle collector could free them. A weak reference doesn't count, so the child no longer keeps the parent alive. Deleting `root` drops its count to zero, it is freed at once, its `children` list goes with it, and so does `docs`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import gc
import weakref

class Folder:
    def __init__(self, name, parent=None):
        self.name = name
        self._parent = weakref.ref(parent) if parent is not None else None
        self.children = []
        if parent is not None:
            parent.children.append(self)

    @property
    def parent(self):
        return self._parent() if self._parent is not None else None

gc.disable()
root = Folder("root")
docs = Folder("docs", root)
print("parent:", docs.parent.name)
watch = weakref.ref(docs)
del root, docs
print("still alive:", watch() is not None)
gc.enable()
```

A `property` (NB21) hides the weak reference, so callers still write `docs.parent` and the fix doesn't ripple through the codebase. The rule: owners hold strong references downwards, back-links are weak. The standard library's `xml.etree` and many GUI toolkits do the same.

</details>

In [ ]:
import gc
import weakref

class Folder:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = parent
        self.children = []
        if parent is not None:
            parent.children.append(self)

gc.disable()
root = Folder("root")
docs = Folder("docs", root)
print("parent:", docs.parent.name)
watch = weakref.ref(docs)
del root, docs
print("still alive:", watch() is not None)
gc.enable()


### NB29-Q22 · L4 · array vs list · Write

Compare a list of 10,000 floats with an `array.array("d", ...)` holding
the same values. Print whether their **shallow** `sys.getsizeof` sizes
are within 10% of each other, whether the list's **deep** size (the
list plus every float in it) is more than 3 times the array's size,
and whether both sum to the same total.

```text
shallow sizes within 10%: True
list deep size > 3x array: True
same total: True
```

**Starting code (already in the cell below):**

```python
import array
import sys

prices = [i * 0.5 for i in range(10_000)]
packed = array.array("d", prices)
```

<details><summary><b>Hint</b></summary>

The shallow ratio is `sys.getsizeof(prices) / sys.getsizeof(packed)`. For the deep size, add `sys.getsizeof(p)` for every `p` in `prices`.

</details>

<details><summary><b>Expected output</b></summary>

```text
shallow sizes within 10%: True
list deep size > 3x array: True
same total: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
list_shallow = sys.getsizeof(prices)
array_size = sys.getsizeof(packed)
list_deep = list_shallow
for p in prices:
    list_deep += sys.getsizeof(p)
print("shallow sizes within 10%:", 0.9 < list_shallow / array_size < 1.1)
print("list deep size > 3x array:", list_deep > 3 * array_size)
print("same total:", sum(prices) == sum(packed))
```

Shallow, they look the same: the list stores an 8-byte pointer per item, the array stores the 8-byte number itself. But each pointer leads to a separate 24-byte float object, which `getsizeof` doesn't count. The array has no per-item objects at all, so its true footprint is about a quarter of the list's.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
list_deep = sys.getsizeof(prices) + sum(map(sys.getsizeof, prices))
array_size = sys.getsizeof(packed)
print("shallow sizes within 10%:", 0.9 < sys.getsizeof(prices) / array_size < 1.1)
print("list deep size > 3x array:", list_deep > 3 * array_size)
print("same total:", sum(prices) == sum(packed))
```

`array` is the standard-library answer for large homogeneous numbers. For real number crunching, seniors use NumPy, which stores numbers the same packed way and does the maths in C. Every deep-size shortcut double-counts shared objects, so they use `tracemalloc` when it matters.

</details>

In [ ]:
import array
import sys

prices = [i * 0.5 for i in range(10_000)]
packed = array.array("d", prices)

# your code here


---

## L5 · Hard

### NB29-Q23 · L5 · weakref · Write

Build an image cache that **doesn't keep images alive**. Write
`get_image(name)` using a `weakref.WeakValueDictionary` called `cache`:
return the cached image if it's still alive, otherwise call
`load_image` and cache the result. Then run this scenario:

1. `a = get_image("logo")`, `b = get_image("logo")`: print `a is b` and `loads`.
2. `del a, b`: print how many entries the cache holds.
3. `c = get_image("logo")`: print `loads`.

```text
True 1
cached: 0
loads: 2
```

**Starting code (already in the cell below):**

```python
import weakref

class Image:
    def __init__(self, name):
        self.name = name

loads = 0

def load_image(name):
    global loads
    loads += 1
    return Image(name)
```

<details><summary><b>Hint</b></summary>

`cache.get(name)` returns `None` when the entry has gone. Keep the new image in a local variable *before* returning it: if nothing else references it, a weak dictionary entry disappears immediately.

</details>

<details><summary><b>Expected output</b></summary>

```text
True 1
cached: 0
loads: 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
cache = weakref.WeakValueDictionary()

def get_image(name):
    image = cache.get(name)
    if image is None:
        image = load_image(name)
        cache[name] = image
    return image

a = get_image("logo")
b = get_image("logo")
print(a is b, loads)
del a, b
print("cached:", len(cache))
c = get_image("logo")
print("loads:", loads)
```

A `WeakValueDictionary` holds its values weakly: an entry lives only as long as something *else* references the value. While `a` and `b` exist the image is shared and loaded once. After `del`, the last strong reference goes, the image is freed, and its entry vanishes, so the next call loads it again. The trap: `cache[name] = load_image(name)` followed by `return cache[name]` can fail, because nothing holds the new image between those two lines.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class ImageCache:
    def __init__(self, loader):
        self._loader = loader
        self._items = weakref.WeakValueDictionary()

    def get(self, name):
        if (image := self._items.get(name)) is None:
            image = self._items[name] = self._loader(name)
        return image

    def __len__(self):
        return len(self._items)

cache = ImageCache(load_image)
a = cache.get("logo")
b = cache.get("logo")
print(a is b, loads)
del a, b
print("cached:", len(cache))
c = cache.get("logo")
print("loads:", loads)
```

Wrapping the cache in a class hides the weak-reference detail and injects the loader, so tests can pass a fake (NB28). The chained assignment keeps `image` as a strong local while it is stored. Compare `lru_cache` (Q14): that one *does* keep results alive, which is right for small values and wrong for big images.

</details>

In [ ]:
import weakref

class Image:
    def __init__(self, name):
        self.name = name

loads = 0

def load_image(name):
    global loads
    loads += 1
    return Image(name)

# your code here


### NB29-Q24 · L5 · cProfile · Write

`convert` looks like the slow part should be `parse`, with all its
splitting. Don't guess: profile **one** call of `convert(lines)` and
print the three functions defined here with the most calls, most first
(ties in name order). Skip built-ins, whose filename in the stats key is
`"~"`. Then print which function is the hotspot.

```text
matches 900
parse 300
rate_for 300
hotspot: matches
```

**Starting code (already in the cell below):**

```python
import cProfile
import pstats

RATES = [("USD", 1.0), ("EUR", 1.08), ("GBP", 1.27), ("JPY", 0.0067),
         ("CHF", 1.13), ("CAD", 0.73)]

def matches(entry, currency):
    return entry[0] == currency

def rate_for(currency):
    for entry in RATES:
        if matches(entry, currency):
            return entry[1]

def parse(line):
    currency, amount = line.split(":")
    return currency.strip(), float(amount)

def convert(lines):
    total = 0.0
    for line in lines:
        currency, amount = parse(line)
        total += amount * rate_for(currency)
    return round(total, 2)

lines = ["USD: 10", "EUR: 20", "GBP: 5", "CHF: 8", "JPY: 900"] * 60
```

<details><summary><b>Hint</b></summary>

Collect `(name, calls)` pairs from `pstats.Stats(profiler).stats`, skipping keys whose filename is `"~"`. Sort by calls descending, then name: a `key=` returning `(-calls, name)` does both.

</details>

<details><summary><b>Expected output</b></summary>

```text
matches 900
parse 300
rate_for 300
hotspot: matches
```

</details>

<details><summary><b>Solution</b></summary>

```python
profiler = cProfile.Profile()
profiler.enable()
convert(lines)
profiler.disable()

counts = []
for (filename, line, name), numbers in pstats.Stats(profiler).stats.items():
    if filename != "~":
        counts.append((name, numbers[1]))
counts.sort(key=lambda pair: (-pair[1], pair[0]))
for name, calls in counts[:3]:
    print(name, calls)
print("hotspot:", counts[0][0])
```

`parse` runs once per line (300 calls), and so does `rate_for`. But `rate_for` scans the `RATES` list, calling `matches` once per entry it passes: 1, 2, 3, 5 and 4 times for the five currencies, so 15 calls per group of five lines, 900 in total. The real cost is a linear search hidden in a helper, which no amount of tuning `parse` would fix. Measure first, then optimise what the profile shows.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with cProfile.Profile() as profiler:
    convert(lines)

stats = pstats.Stats(profiler).stats
counts = sorted(((name, numbers[1]) for (filename, _, name), numbers in stats.items()
                 if filename != "~" and name != "__exit__"),
                key=lambda pair: (-pair[1], pair[0]))
for name, calls in counts[:3]:
    print(name, calls)
print("hotspot:", counts[0][0])

RATE_TABLE = dict(RATES)  # the fix the profile points to: O(1) lookups
```

With `with cProfile.Profile()` the profiler also records its own `__exit__`, so the senior filters it out. The last line is the actual fix: a dict makes `rate_for` O(1) and `matches` disappears. Seniors profile, fix the top item, then profile again, because the next hotspot is often somewhere else.

</details>

In [ ]:
import cProfile
import pstats

RATES = [("USD", 1.0), ("EUR", 1.08), ("GBP", 1.27), ("JPY", 0.0067),
         ("CHF", 1.13), ("CAD", 0.73)]

def matches(entry, currency):
    return entry[0] == currency

def rate_for(currency):
    for entry in RATES:
        if matches(entry, currency):
            return entry[1]

def parse(line):
    currency, amount = line.split(":")
    return currency.strip(), float(amount)

def convert(lines):
    total = 0.0
    for line in lines:
        currency, amount = parse(line)
        total += amount * rate_for(currency)
    return round(total, 2)

lines = ["USD: 10", "EUR: 20", "GBP: 5", "CHF: 8", "JPY: 900"] * 60

# your code here


### NB29-Q25 · L5 · __slots__ · Write

Give `Sensor` slots for `name` and `reading` **and** keep it usable with
weak references, then track live sensors in a
`weakref.WeakValueDictionary` called `registry` keyed by name. Create
`s1` (`"boiler"`, `71.5`) and `s2` (`"attic"`, `18.0`), register both,
then print:

```text
has __dict__: False
registered: 2
after removing s1: 1
['attic']
```

**Starting code (already in the cell below):**

```python
import weakref
```

<details><summary><b>Hint</b></summary>

`__slots__` removes the slot weak references need. Add `"__weakref__"` to the slots to get it back.

</details>

<details><summary><b>Expected output</b></summary>

```text
has __dict__: False
registered: 2
after removing s1: 1
['attic']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Sensor:
    __slots__ = ("name", "reading", "__weakref__")

    def __init__(self, name, reading):
        self.name = name
        self.reading = reading

registry = weakref.WeakValueDictionary()
s1 = Sensor("boiler", 71.5)
s2 = Sensor("attic", 18.0)
registry[s1.name] = s1
registry[s2.name] = s2
print("has __dict__:", hasattr(s1, "__dict__"))
print("registered:", len(registry))
del s1
print("after removing s1:", len(registry))
print(sorted(registry.keys()))
```

`__slots__` saves memory by dropping the per-instance `__dict__`, and it also drops `__weakref__`, so without it `WeakValueDictionary` raises `TypeError`. Listing `"__weakref__"` adds back only that one slot. The registry never keeps a sensor alive: `del s1` removes the last strong reference and the entry disappears.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass

@dataclass(slots=True, weakref_slot=True)
class Sensor:
    name: str
    reading: float

registry = weakref.WeakValueDictionary()
s1 = Sensor("boiler", 71.5)
s2 = Sensor("attic", 18.0)
registry.update({s.name: s for s in (s1, s2)})
print("has __dict__:", hasattr(s1, "__dict__"))
print("registered:", len(registry))
del s1
print("after removing s1:", len(registry))
print(sorted(registry))
```

`@dataclass(slots=True, weakref_slot=True)` (NB21) generates both slots plus `__init__` and `__repr__`. Slots pay off when you create many thousands of small objects; for a handful of objects, the plain class is simpler.

</details>

In [ ]:
import weakref

# your code here


### NB29-Q26 · L5 · Generators and memory · Fix

This pipeline was meant to stream 200,000 sensor readings in constant
memory, yet its memory check fails. Find the step that defeats the
laziness and fix it, so the output becomes:

```text
total: 149700000
low memory: True
```

**Buggy code (copied into the cell below):**

```python
import tracemalloc

def readings(n):
    for i in range(n):
        yield (i * 37) % 1000

def above(values, limit):
    return [v for v in values if v > limit]

tracemalloc.start()
total = sum(v * 2 for v in above(readings(200_000), 500))
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print("total:", total)
print("low memory:", peak < 100_000)
```

<details><summary><b>Hint</b></summary>

One generator in a chain that builds a list makes the whole chain hold everything at once. Which step uses square brackets?

</details>

<details><summary><b>Expected output</b></summary>

```text
total: 149700000
low memory: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
import tracemalloc

def readings(n):
    for i in range(n):
        yield (i * 37) % 1000

def above(values, limit):
    return (v for v in values if v > limit)

tracemalloc.start()
total = sum(v * 2 for v in above(readings(200_000), 500))
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print("total:", total)
print("low memory:", peak < 100_000)
```

`above` built a list of about 100,000 readings before `sum` saw any of them, so the peak was the list plus its ints: megabytes. Swapping the square brackets for round ones returns a generator, and now every stage passes one value along at a time. A lazy pipeline is only as lazy as its greediest step.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import tracemalloc

def readings(n):
    for i in range(n):
        yield (i * 37) % 1000

def above(values, limit):
    for v in values:
        if v > limit:
            yield v

tracemalloc.start()
try:
    total = sum(v * 2 for v in above(readings(200_000), 500))
    peak = tracemalloc.get_traced_memory()[1]
finally:
    tracemalloc.stop()
print("total:", total)
print("low memory:", peak < 100_000)
```

A `yield` loop (NB22) makes the laziness explicit and gives room for more logic later. A memory assertion like `peak < 100_000` makes a good regression test: it fails loudly the day someone adds a `list()` somewhere in the chain.

</details>

In [ ]:
import tracemalloc

def readings(n):
    for i in range(n):
        yield (i * 37) % 1000

def above(values, limit):
    return [v for v in values if v > limit]

tracemalloc.start()
total = sum(v * 2 for v in above(readings(200_000), 500))
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print("total:", total)
print("low memory:", peak < 100_000)


---

## Mini-project

Pull together everything from this notebook.

## NB29-P · L5 · Mini-project: Report Speed-up · Write

`slow_report` produces a revenue report for 2,000 orders across 300
products, and it is slow. Measure it, write `fast_report` that gives
the **identical** text, and prove the improvement with counts, not
timings.

1. Profile one call of `slow_report(orders)` with `cProfile` and read
   how many times `find_price` was called. Note `steps` (catalogue
   entries scanned) afterwards as the slow cost.
2. Write `fast_report(orders)`: build a dict from `catalogue` once,
   count one step per price lookup in a global `fast_steps`, and
   build the text with `"".join` instead of `+=`. Keep the order of
   the additions the same, so the float totals match exactly.
3. Print this summary (the first report line comes from the fast
   report):

```text
find_price calls: 2000
Products in report: 300
First line: P000: 7.00
Slow lookup steps: 299400
Fast lookup steps: 2000
Same report: True
At least 100x fewer steps: True
```

**Starting code (already in the cell below):**

```python
import cProfile
import pstats

catalogue = [(f"P{i:03d}", round(1 + i * 0.25, 2)) for i in range(300)]
orders = [(f"P{(i * 7) % 300:03d}", i % 5 + 1) for i in range(2000)]

steps = 0

def find_price(code):
    global steps
    for item_code, price in catalogue:
        steps += 1
        if item_code == code:
            return price
    return None

def slow_report(orders):
    totals = {}
    for code, qty in orders:
        totals[code] = totals.get(code, 0) + find_price(code) * qty
    text = ""
    for code in sorted(totals):
        text += f"{code}: {totals[code]:.2f}\n"
    return text
```

<details><summary><b>Hint</b></summary>

Profile with `enable()`/`disable()` and look up `find_price` in `pstats.Stats(profiler).stats` as in Q11. In `fast_report`, `prices = dict(catalogue)` replaces the scan. Collect lines in a list and join with `""`, keeping the `\n` on each line.

</details>

<details><summary><b>Expected output</b></summary>

```text
find_price calls: 2000
Products in report: 300
First line: P000: 7.00
Slow lookup steps: 299400
Fast lookup steps: 2000
Same report: True
At least 100x fewer steps: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
profiler = cProfile.Profile()
profiler.enable()
slow_text = slow_report(orders)
profiler.disable()
for (filename, line, name), numbers in pstats.Stats(profiler).stats.items():
    if name == "find_price":
        print("find_price calls:", numbers[1])
slow_steps = steps

fast_steps = 0

def fast_report(orders):
    global fast_steps
    prices = dict(catalogue)
    totals = {}
    for code, qty in orders:
        fast_steps += 1
        totals[code] = totals.get(code, 0) + prices[code] * qty
    lines = []
    for code in sorted(totals):
        lines.append(f"{code}: {totals[code]:.2f}\n")
    return "".join(lines)

fast_text = fast_report(orders)
report_lines = fast_text.splitlines()
print("Products in report:", len(report_lines))
print("First line:", report_lines[0])
print("Slow lookup steps:", slow_steps)
print("Fast lookup steps:", fast_steps)
print("Same report:", slow_text == fast_text)
print("At least 100x fewer steps:", slow_steps >= 100 * fast_steps)
```

The profile shows 2,000 calls to `find_price`, one per order, and each scans the catalogue from the start: about half of 300 entries on average, 299,400 steps in all. Building a dict once turns each lookup into one hash step, so the work drops from O(orders x products) to O(orders). Using `join` removes the quadratic string building. Comparing the two texts proves the speed-up didn't change the answer, which is the first thing to check after any optimisation.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict

with cProfile.Profile() as profiler:
    slow_text = slow_report(orders)
profile = pstats.Stats(profiler).get_stats_profile()
print("find_price calls:", profile.func_profiles["find_price"].ncalls)
slow_steps = steps

fast_steps = 0
PRICES = dict(catalogue)

def fast_report(orders):
    global fast_steps
    totals = defaultdict(int)
    for code, qty in orders:
        fast_steps += 1
        totals[code] += PRICES[code] * qty
    return "".join(f"{code}: {totals[code]:.2f}\n" for code in sorted(totals))

fast_text = fast_report(orders)
report_lines = fast_text.splitlines()
print("Products in report:", len(report_lines))
print("First line:", report_lines[0])
print("Slow lookup steps:", slow_steps)
print("Fast lookup steps:", fast_steps)
print("Same report:", slow_text == fast_text)
print("At least 100x fewer steps:", slow_steps >= 100 * fast_steps)
```

The senior builds the price table once at module level (it doesn't change between reports), uses `defaultdict(int)` (NB15) for the running totals, and reads the call count by name with `get_stats_profile()`. The workflow is the real lesson: measure, change one thing, prove the output is identical, measure again. For money, the totals would be `Decimal` (NB16) or whole cents.

</details>

In [ ]:
import cProfile
import pstats

catalogue = [(f"P{i:03d}", round(1 + i * 0.25, 2)) for i in range(300)]
orders = [(f"P{(i * 7) % 300:03d}", i % 5 + 1) for i in range(2000)]

steps = 0

def find_price(code):
    global steps
    for item_code, price in catalogue:
        steps += 1
        if item_code == code:
            return price
    return None

def slow_report(orders):
    totals = {}
    for code, qty in orders:
        totals[code] = totals.get(code, 0) + find_price(code) * qty
    text = ""
    for code in sorted(totals):
        text += f"{code}: {totals[code]:.2f}\n"
    return text

# your code here


---

## Done

Next up: **NB30 · Networking HTTP APIs** in `advanced/`.